# Reto proyecto: Genera imágenes con Stable Diffusion

**Nombre y apellidos:** Escribe aquí tu nombre y apellidos  
**Concepto elegido:** Flores decorativas para ecommerce  
**Modelo principal:** `stabilityai/stable-diffusion-2-1-base`  
**Técnicas utilizadas:** text-to-image, image-to-image, prompts negativos, semillas, distintos schedulers y variación de parámetros.

Este notebook está preparado para ejecutarse en Google Colab o en un entorno local con Python. El objetivo es generar imágenes coherentes de un concepto visual, organizar un conjunto de datos en `custom_dataset`, experimentar con prompts y parámetros, y crear variaciones de una imagen base mediante image-to-image.


## 1. Instalación y preparación del entorno

Esta celda instala las librerías necesarias. En Google Colab se puede ejecutar directamente. Si ya tienes las dependencias instaladas en tu entorno local, puedes omitirla.


In [ ]:
# Instalar dependencias principales
# En Google Colab: ejecuta esta celda.
# En local: puedes instalar lo mismo desde terminal.
!pip install -q diffusers transformers accelerate safetensors torch pillow matplotlib


## 2. Importación de librerías, rutas y comprobación de GPU

Se comprueba si hay GPU disponible. Si no la hay, el notebook también puede ejecutarse en CPU, aunque la generación será mucho más lenta.


In [ ]:
import os
import math
import random
import csv
from pathlib import Path

import torch
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display

from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionImg2ImgPipeline,
    EulerAncestralDiscreteScheduler,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
)

PROJECT_DIR = Path('.')
DATASET_DIR = PROJECT_DIR / 'custom_dataset'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
TXT2IMG_DIR = OUTPUT_DIR / 'txt2img'
IMG2IMG_DIR = OUTPUT_DIR / 'img2img'

for path in [DATASET_DIR, TXT2IMG_DIR, IMG2IMG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Dispositivo usado: {device}')
print(f'Tipo de dato usado: {dtype}')
if device == 'cuda':
    print(f'GPU detectada: {torch.cuda.get_device_name(0)}')


## 3. Preparación del conjunto de datos `custom_dataset`

El enunciado pide una carpeta `custom_dataset` con al menos 10 imágenes representativas del concepto elegido.

Para que el proyecto sea reproducible incluso si una URL externa falla, esta celda verifica si ya existen imágenes en `custom_dataset`. Si no existen, crea 10 imágenes base del concepto **flores decorativas**, con variaciones de perspectiva, estilo y contexto. En una entrega real, también puedes sustituir estas imágenes por fotografías descargadas manualmente, manteniendo la misma carpeta.


In [ ]:
def create_flower_reference_image(idx: int, size: int = 768) -> Image.Image:
    """Crea una imagen base representativa del concepto 'flores decorativas'."""
    rnd = random.Random(1000 + idx)
    backgrounds = [
        (245, 245, 240), (238, 245, 255), (250, 240, 245), (242, 248, 238),
        (250, 248, 238), (236, 236, 245), (245, 239, 229), (230, 245, 242),
        (248, 245, 250), (240, 240, 235)
    ]
    img = Image.new('RGB', (size, size), backgrounds[idx % len(backgrounds)])
    draw = ImageDraw.Draw(img, 'RGBA')

    for y in range(size):
        alpha = int(22 * y / size)
        draw.line([(0, y), (size, y)], fill=(255, 255, 255, alpha))

    vase_x = size // 2 + rnd.randint(-40, 40)
    vase_top = int(size * 0.55) + rnd.randint(-25, 25)
    vase_bottom = int(size * 0.88)
    vase_w_top = rnd.randint(110, 170)
    vase_w_bottom = rnd.randint(160, 230)

    draw.polygon(
        [
            (vase_x - vase_w_top // 2, vase_top),
            (vase_x + vase_w_top // 2, vase_top),
            (vase_x + vase_w_bottom // 2, vase_bottom),
            (vase_x - vase_w_bottom // 2, vase_bottom),
        ],
        fill=(205, 225, 240, 170),
        outline=(160, 175, 190, 160),
    )
    draw.ellipse([vase_x - vase_w_top // 2, vase_top - 10, vase_x + vase_w_top // 2, vase_top + 22], fill=(210, 230, 245, 120))

    flower_colors = [
        (240, 90, 120, 235), (255, 185, 70, 235), (170, 105, 220, 235),
        (255, 230, 90, 235), (245, 120, 180, 235), (120, 180, 255, 235),
        (250, 250, 250, 235), (220, 75, 95, 235)
    ]
    leaf_color = (80, 145, 85, 220)
    n_flowers = rnd.randint(6, 11)

    for i in range(n_flowers):
        angle = (-45 + i * (90 / max(1, n_flowers - 1)) + rnd.randint(-16, 16)) * math.pi / 180
        length = rnd.randint(210, 360)
        start = (vase_x + rnd.randint(-25, 25), vase_top + rnd.randint(0, 25))
        end = (int(start[0] + length * math.sin(angle)), int(start[1] - length * math.cos(angle)))

        draw.line([start, end], fill=(70, 135, 75, 230), width=rnd.randint(5, 8))
        for _ in range(rnd.randint(1, 2)):
            t = rnd.random() * 0.7 + 0.15
            lx = int(start[0] + (end[0] - start[0]) * t)
            ly = int(start[1] + (end[1] - start[1]) * t)
            lw = rnd.randint(30, 50)
            lh = rnd.randint(12, 22)
            draw.ellipse([lx - lw, ly - lh, lx + lw, ly + lh], fill=leaf_color)

        color = flower_colors[(idx + i) % len(flower_colors)]
        cx, cy = end
        petal_count = rnd.choice([6, 7, 8, 10])
        radius = rnd.randint(34, 52)
        for p in range(petal_count):
            theta = 2 * math.pi * p / petal_count + rnd.uniform(-0.08, 0.08)
            px = cx + int(math.cos(theta) * radius * 0.62)
            py = cy + int(math.sin(theta) * radius * 0.62)
            ew = rnd.randint(24, 42)
            eh = rnd.randint(34, 52)
            draw.ellipse([px - ew // 2, py - eh // 2, px + ew // 2, py + eh // 2], fill=color)
        draw.ellipse([cx - 15, cy - 15, cx + 15, cy + 15], fill=(120, 80, 35, 245))

    draw.ellipse([vase_x - vase_w_bottom // 2 - 25, vase_bottom - 8, vase_x + vase_w_bottom // 2 + 25, vase_bottom + 25], fill=(120, 120, 120, 45))
    if idx % 3 == 0:
        draw.rectangle([40, size - 160, 240, size - 130], fill=(210, 200, 185, 120))
    if idx % 4 == 1:
        draw.ellipse([size - 180, 85, size - 95, 170], fill=(255, 255, 255, 180), outline=(190, 190, 190, 120))
    if idx % 4 == 2:
        draw.rectangle([0, 0, 30, size], fill=(210, 210, 210, 110))

    return img.filter(ImageFilter.SMOOTH_MORE)

existing_images = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']])

if len(existing_images) < 10:
    print('No se han encontrado 10 imágenes. Creando dataset base reproducible en custom_dataset...')
    metadata_rows = []
    for i in range(10):
        img = create_flower_reference_image(i)
        filename = f'flower_reference_{i+1:02d}.png'
        img.save(DATASET_DIR / filename)
        metadata_rows.append({
            'filename': filename,
            'concept': 'flores decorativas para ecommerce',
            'variation': ['frontal', 'ángulo lateral', 'estilo macro', 'estudio minimalista', 'contexto lifestyle'][i % 5],
            'description': f'Imagen base {i+1} con variación de perspectiva, fondo y estilo.'
        })
    with open(DATASET_DIR / 'metadata.csv', 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['filename', 'concept', 'variation', 'description'])
        writer.writeheader()
        writer.writerows(metadata_rows)
else:
    print('Dataset existente detectado. Se usará el contenido actual de custom_dataset.')

dataset_images = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in ['.png', '.jpg', '.jpeg', '.webp']])
print(f'Número de imágenes en custom_dataset: {len(dataset_images)}')
for img_path in dataset_images[:10]:
    print('-', img_path)


## 4. Visualización del dataset

Se muestra una cuadrícula con las 10 imágenes base para comprobar que el conjunto de datos está correctamente organizado y contiene diferentes variantes.


In [ ]:
def make_image_grid(images, rows, cols, resize_to=220):
    processed = []
    for img in images:
        if isinstance(img, (str, Path)):
            img = Image.open(img).convert('RGB')
        else:
            img = img.convert('RGB')
        img = img.copy()
        img.thumbnail((resize_to, resize_to))
        canvas = Image.new('RGB', (resize_to, resize_to), 'white')
        x = (resize_to - img.width) // 2
        y = (resize_to - img.height) // 2
        canvas.paste(img, (x, y))
        processed.append(canvas)

    grid = Image.new('RGB', (cols * resize_to, rows * resize_to), 'white')
    for i, img in enumerate(processed[: rows * cols]):
        x = (i % cols) * resize_to
        y = (i // cols) * resize_to
        grid.paste(img, (x, y))
    return grid

dataset_grid = make_image_grid(dataset_images[:10], rows=2, cols=5, resize_to=220)
dataset_grid.save(OUTPUT_DIR / 'custom_dataset_grid.png')
display(dataset_grid)


## 5. Carga del modelo Stable Diffusion y configuración del scheduler

Se utiliza el modelo pedido en el enunciado: `stabilityai/stable-diffusion-2-1-base`.

El scheduler principal será `EulerAncestralDiscreteScheduler`, y después se comparará con otros schedulers para analizar su influencia en la generación.


In [ ]:
MODEL_ID = 'stabilityai/stable-diffusion-2-1-base'

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
)

pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

pipe.enable_attention_slicing()
try:
    pipe.enable_vae_slicing()
except Exception:
    pass

print('Modelo cargado correctamente.')
print('Scheduler inicial:', pipe.scheduler.__class__.__name__)


## 6. Definición de prompts y parámetros de inferencia

Se incluyen dos prompts principales, con estilos diferentes, y un `negative_prompt` para evitar baja calidad, deformaciones, texto no deseado o marcas de agua.


In [ ]:
negative_prompt = (
    'low quality, blurry, distorted, deformed, bad composition, '
    'extra objects, text, watermark, logo, jpeg artifacts, unrealistic proportions, '
    'duplicate flowers, cropped product, messy background'
)

prompts = [
    {
        'name': 'ramo_premium_fondo_blanco',
        'prompt': (
            'A high-resolution ecommerce product photo of a premium decorative flower bouquet '
            'in a transparent glass vase, white studio background, soft natural lighting, '
            'sharp focus, realistic petals, elegant composition, professional product photography'
        ),
        'guidance_scale': 7.5,
        'num_inference_steps': 30,
        'seed': 42,
        'scheduler': 'EulerAncestral'
    },
    {
        'name': 'flores_lifestyle_mesa_madera',
        'prompt': (
            'A realistic lifestyle product photo of colorful spring flowers in a ceramic vase '
            'on a wooden table, warm morning light, cozy home decor, shallow depth of field, '
            'high detail, natural colors, ecommerce catalog style'
        ),
        'guidance_scale': 8.5,
        'num_inference_steps': 35,
        'seed': 123,
        'scheduler': 'EulerAncestral'
    },
]

if device == 'cpu':
    for item in prompts:
        item['num_inference_steps'] = min(item['num_inference_steps'], 12)

prompts


## 7. Generación text-to-image

Esta sección genera imágenes desde texto, guarda los resultados y crea una cuadrícula comparativa.


In [ ]:
def set_scheduler(pipeline, scheduler_name: str):
    scheduler_name = scheduler_name.lower()
    if scheduler_name == 'eulerancestral':
        pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config)
    elif scheduler_name == 'ddim':
        pipeline.scheduler = DDIMScheduler.from_config(pipeline.scheduler.config)
    elif scheduler_name == 'dpm':
        pipeline.scheduler = DPMSolverMultistepScheduler.from_config(pipeline.scheduler.config)
    else:
        raise ValueError(f'Scheduler no reconocido: {scheduler_name}')
    return pipeline

generated_images = []
generation_log = []

for cfg in prompts:
    pipe = set_scheduler(pipe, cfg['scheduler'])
    generator = torch.Generator(device=device).manual_seed(cfg['seed'])
    image = pipe(
        prompt=cfg['prompt'],
        negative_prompt=negative_prompt,
        guidance_scale=cfg['guidance_scale'],
        num_inference_steps=cfg['num_inference_steps'],
        generator=generator,
        width=512,
        height=512,
    ).images[0]
    output_path = TXT2IMG_DIR / f"{cfg['name']}_{cfg['scheduler']}_seed{cfg['seed']}.png"
    image.save(output_path)
    generated_images.append(image)
    generation_log.append({
        'file': str(output_path),
        'prompt': cfg['prompt'],
        'scheduler': cfg['scheduler'],
        'guidance_scale': cfg['guidance_scale'],
        'num_inference_steps': cfg['num_inference_steps'],
        'seed': cfg['seed'],
    })
    print(f'Imagen guardada: {output_path}')

txt2img_grid = make_image_grid(generated_images, rows=1, cols=len(generated_images), resize_to=300)
txt2img_grid.save(TXT2IMG_DIR / 'txt2img_grid.png')
display(txt2img_grid)


## 8. Experimentación con distintos schedulers

Aquí se genera el mismo concepto con distintos schedulers para comparar cómo afectan a la imagen final. Se prueban `EulerAncestral`, `DDIM` y `DPM`.


In [ ]:
scheduler_experiments = [
    {'scheduler': 'EulerAncestral', 'seed': 77, 'guidance_scale': 7.5, 'num_inference_steps': 28 if device == 'cuda' else 10},
    {'scheduler': 'DDIM', 'seed': 77, 'guidance_scale': 7.5, 'num_inference_steps': 28 if device == 'cuda' else 10},
    {'scheduler': 'DPM', 'seed': 77, 'guidance_scale': 7.5, 'num_inference_steps': 28 if device == 'cuda' else 10},
]

scheduler_prompt = (
    'A professional ecommerce photo of a luxury bouquet of pink and white flowers '
    'in a minimalist glass vase, white background, realistic lighting, high detail, '
    'centered product, clean catalog photography'
)

scheduler_images = []
for exp in scheduler_experiments:
    pipe = set_scheduler(pipe, exp['scheduler'])
    generator = torch.Generator(device=device).manual_seed(exp['seed'])
    image = pipe(
        prompt=scheduler_prompt,
        negative_prompt=negative_prompt,
        guidance_scale=exp['guidance_scale'],
        num_inference_steps=exp['num_inference_steps'],
        generator=generator,
        width=512,
        height=512,
    ).images[0]
    output_path = TXT2IMG_DIR / f"scheduler_test_{exp['scheduler']}_seed{exp['seed']}.png"
    image.save(output_path)
    scheduler_images.append(image)
    print(f'Guardado: {output_path}')

scheduler_grid = make_image_grid(scheduler_images, rows=1, cols=3, resize_to=260)
scheduler_grid.save(TXT2IMG_DIR / 'scheduler_comparison_grid.png')
display(scheduler_grid)


## 9. Image-to-image: generación de variaciones a partir de una imagen base

Se toma una imagen de `custom_dataset` y se usa como imagen inicial. La variación se guía con un prompt que cambia el estilo y el contexto visual del producto.


In [ ]:
base_image_path = dataset_images[0]
init_image = Image.open(base_image_path).convert('RGB').resize((512, 512))
print('Imagen base usada:', base_image_path)
display(init_image)


In [ ]:
img2img_pipe = StableDiffusionImg2ImgPipeline(**pipe.components)
img2img_pipe = img2img_pipe.to(device)
img2img_pipe.enable_attention_slicing()
try:
    img2img_pipe.enable_vae_slicing()
except Exception:
    pass

print('Pipeline Image-to-Image creado correctamente.')


In [ ]:
img2img_prompt = (
    'A futuristic luxury version of a decorative flower bouquet in a transparent vase, '
    'neon pink and blue accents, dark elegant background, premium ecommerce product photo, '
    'realistic petals, dramatic studio lighting, high detail'
)

img2img_experiments = [
    {'name': 'variacion_suave_euler', 'scheduler': 'EulerAncestral', 'strength': 0.35, 'guidance_scale': 7.5, 'num_inference_steps': 30 if device == 'cuda' else 10, 'seed': 2024},
    {'name': 'variacion_creativa_ddim', 'scheduler': 'DDIM', 'strength': 0.65, 'guidance_scale': 8.5, 'num_inference_steps': 35 if device == 'cuda' else 12, 'seed': 2025},
]

img2img_results = []
for exp in img2img_experiments:
    img2img_pipe = set_scheduler(img2img_pipe, exp['scheduler'])
    generator = torch.Generator(device=device).manual_seed(exp['seed'])
    result = img2img_pipe(
        prompt=img2img_prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        strength=exp['strength'],
        guidance_scale=exp['guidance_scale'],
        num_inference_steps=exp['num_inference_steps'],
        generator=generator,
    ).images[0]
    output_path = IMG2IMG_DIR / f"{exp['name']}_{exp['scheduler']}_strength{exp['strength']}.png"
    result.save(output_path)
    img2img_results.append(result)
    print(f'Variación guardada: {output_path}')

img2img_grid = make_image_grid([init_image] + img2img_results, rows=1, cols=3, resize_to=260)
img2img_grid.save(IMG2IMG_DIR / 'img2img_grid.png')
display(img2img_grid)


## 10. Resumen de decisiones y análisis

**Preparación del dataset:** se ha creado/organizado la carpeta `custom_dataset` con 10 imágenes del concepto elegido: flores decorativas para ecommerce. Las imágenes representan variaciones de ángulo, color, fondo y estilo.

**Modelo utilizado:** se ha usado `stabilityai/stable-diffusion-2-1-base`, un modelo preentrenado de Stable Diffusion adecuado para generación text-to-image e image-to-image.

**Scheduler principal:** se ha configurado `EulerAncestralDiscreteScheduler` como scheduler principal. Además, se han comparado variantes con `DDIMScheduler` y `DPMSolverMultistepScheduler`.

**Prompts:** se han usado dos prompts distintos: uno de fotografía ecommerce con fondo blanco y otro lifestyle con mesa de madera y luz cálida.

**Negative prompt:** se ha usado un prompt negativo para reducir errores habituales como baja calidad, distorsiones, texto, marcas de agua, fondos caóticos o proporciones irreales.

**Parámetros de inferencia:**
- `guidance_scale`: controla cuánto sigue la imagen el prompt.
- `num_inference_steps`: más pasos suelen mejorar el detalle, aunque aumentan el tiempo de ejecución.
- `seed`: permite reproducir resultados concretos.
- `strength` en image-to-image: valores bajos conservan más la imagen original; valores altos permiten cambios más creativos.

**Image-to-image:** la imagen base se carga desde `custom_dataset` y se transforma con un prompt futurista. Se prueban dos niveles de `strength` y dos schedulers para observar diferencias en fidelidad y creatividad.


## 11. Conclusión

El proyecto cumple los requisitos del reto:

- Incluye un conjunto de datos organizado en `custom_dataset` con al menos 10 imágenes.
- Configura el entorno con `diffusers`, `torch`, `transformers`, `accelerate` y `PIL`.
- Comprueba la disponibilidad de GPU.
- Carga el modelo `stabilityai/stable-diffusion-2-1-base`.
- Usa `EulerAncestralDiscreteScheduler` y compara otros schedulers.
- Genera imágenes con al menos dos prompts distintos.
- Aplica `negative_prompt`.
- Implementa image-to-image con una imagen base local.
- Guarda los resultados en carpetas organizadas dentro de `outputs`.
